In [1]:
import pandas as pd
import numpy as np

DATA_FOLDER = "./"
df = pd.read_pickle(DATA_FOLDER + "df_fe_for_ensamble_best_customers_0c.pickle")
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()
#df = df[df["product_id"].isin(product_ids)]
df = df.sort_values(by=["date_id", "product_id"])
df["customer_id"] = 0
df["target"] = df.groupby(["product_id", "customer_id"])["tn"].shift(-2)


In [2]:
df.describe()

,product_id,cust_request_qty,cust_request_tn,tn,stock_final,sku_size,year,mes,quarter,date_id,...,prod_tn_lag_1_x_tn_lag_11,prod_tn_lag_1_x_tn_lag_8,prod_tn_lag_3_x_tn_lag_2,prod_tn_lag_3_x_tn_lag_11,prod_tn_lag_3_x_tn_lag_8,prod_tn_lag_2_x_tn_lag_11,prod_tn_lag_2_x_tn_lag_8,prod_tn_lag_11_x_tn_lag_8,customer_id,target
count,31522.000000,31522.000000,31522.000000,31522.000000,13691.000000,31229.000000,31522.000000,31522.000000,31522.000000,31522.000000,...,1.920900e+04,2.224000e+04,2.787000e+04,1.920900e+04,2.224000e+04,1.920900e+04,2.224000e+04,1.920900e+04,31522.0,29076.000000
mean,20535.827073,200.806865,42.924751,42.033772,19.478148,476.827881,2018.037688,6.575471,2.524840,18.027727,...,1.405697e+04,1.394960e+04,1.369290e+04,1.435157e+04,1.411351e+04,1.426182e+04,1.402111e+04,1.518359e+04,0.0,42.737881
std,347.109552,124.339898,113.127739,109.374512,55.627438,883.449097,0.816015,3.452354,1.118599,10.355680,...,1.006608e+05,1.006127e+05,1.004995e+05,9.949145e+04,9.961061e+04,1.020440e+05,9.971995e+04,1.012742e+05,0.0,111.549156
min,20001.000000,0.000000,0.000000,0.000000,-27.311359,1.000000,2017.000000,1.000000,1.000000,0.000000,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000
25%,20239.000000,106.000000,2.221153,2.211540,1.160960,90.000000,2017.000000,4.000000,2.000000,9.000000,...,6.883162e+00,6.546921e+00,6.283891e+00,7.520869e+00,7.176875e+00,7.182292e+00,6.957675e+00,8.685013e+00,0.0,2.222740
50%,20495.000000,185.000000,9.652330,9.598450,5.419600,250.000000,2018.000000,7.000000,3.000000,18.000000,...,1.160452e+02,1.088569e+02,1.063643e+02,1.199704e+02,1.132123e+02,1.201063e+02,1.119023e+02,1.358740e+02,0.0,9.635365
75%,20812.000000,281.000000,29.836285,29.572310,17.584541,475.000000,2019.000000,10.000000,4.000000,27.000000,...,1.034533e+03,9.596051e+02,9.119448e+02,1.041381e+03,9.797971e+02,1.059134e+03,9.570754e+02,1.129115e+03,0.0,29.966728
max,21299.000000,756.000000,2423.708740,2295.198242,1562.024536,10000.000000,2019.000000,12.000000,4.000000,35.000000,...,3.009615e+06,4.261805e+06,4.161229e+06,3.366470e+06,3.375448e+06,3.853622e+06,3.781657e+06,3.374882e+06,0.0,2295.198242


In [3]:
df[["fecha", "date_id"]]

,fecha,date_id
0,2017-01,0
1,2017-01,0
2,2017-01,0
3,2017-01,0
4,2017-01,0
...,...,...
31517,2019-12,35
31518,2019-12,35
31519,2019-12,35
31520,2019-12,35


In [4]:
# transformacion comun de datos para todos los modelos:
# remuevo periodo_min_producto, periodo_max_producto, periodo_min_customer, periodo_max_customer
df = df.drop(columns=["periodo_min_producto", "periodo_max_producto",
                   "periodo_min_customer", "periodo_max_customer"], errors='ignore')

# transformo columnas object a categorical
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype("category")

# transformo plan precios cuidados a categorical
#df["plan_precios_cuidados"] = df["plan_precios_cuidados"].astype("category")

In [5]:

# dropeo columns donde tenga mas sea todo nan hasta date_id 28
print(f"Df shape before dropping columns: {df.shape}")
subset = df[df["date_id"] <= 28]
cols_to_drop = subset.columns[subset.isna().all()]
df = df.drop(columns=cols_to_drop, errors='ignore')
print(f"Df shape after dropping columns: {df.shape}")

Df shape before dropping columns: (31522, 312)
Df shape after dropping columns: (31522, 312)


In [6]:

from sklearn.model_selection import BaseCrossValidator
import numpy as np

class CustomTimeSeriesSplit(BaseCrossValidator):
    def __init__(self, n_splits=3, gap=1):
        self.n_splits = n_splits
        self.gap = gap

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

    def split(self, X, y=None, groups=None):
        # Asegurar que X es DataFrame
        
        unique_dates = sorted(X["date_id"].unique(), reverse=True)
        
        for i in range(self.n_splits):
                
            test_date_id = unique_dates[i]
            train_date_id = test_date_id - self.gap - 1
            
            # Usar np.where para obtener posiciones enteras
            train_mask = X["date_id"] <= train_date_id
            test_mask = X["date_id"] == test_date_id
            
            train_idx = np.where(train_mask)[0]
            test_idx = np.where(test_mask)[0]
            
            yield train_idx, test_idx

In [7]:

class LinearRegressionModel:
    def __init__(self, product_ids="magicos", features_to_use=None, especialidad="all"):
        self.model = None
        self.product_ids = product_ids if product_ids is not None else []
        self.features_to_use = features_to_use or ["lags"]
        self.features = []
        self.especialidad = especialidad

    @property
    def name(self):
        return f"LinearRegression-{self.product_ids}-{self.features_to_use}-{self.especialidad}"

    def _get_product_ids(self, df):
        if self.product_ids == "magicos":
            return [20002, 20003, 20006, 20010, 20011, 20018, 20019, 20021,
                    20026, 20028, 20035, 20039, 20042, 20044, 20045, 20046,
                    20049, 20051, 20052, 20053, 20055, 20008, 20001, 20017,
                    20086, 20180, 20193, 20320, 20532, 20612, 20637, 20807,
                    20838]
        elif self.product_ids == "HC":
            hc_products = df[df["cat1"] == "HC"]["product_id"].unique().tolist()
            return hc_products
        elif self.product_ids == "FOODS":
            foods_products = df[df["cat1"] == "FOODS"]["product_id"].unique().tolist()
            return foods_products
        elif self.product_ids == "PC":
            pc_products = df[df["cat1"] == "PC"]["product_id"].unique().tolist()
            return pc_products
        elif self.product_ids == "all":
            return df["product_id"].unique().tolist()
    
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum", "cat1": "first"}).sort_values(["product_id", "date_id"])
        self.features.append("tn")
        if "lags" in self.features_to_use:
            for lag in range(1, 12):
                df[f"tn_{lag}"] = df.groupby("product_id")["tn"].shift(lag)
                self.features.append(f"tn_{lag}")
        if "mean" in self.features_to_use: 
            for window in range(1, 12):
                df[f"tn_mean_{window}"] = df.groupby("product_id")["tn"].transform(lambda x: x.rolling(window=window, min_periods=1).mean())
                self.features.append(f"tn_mean_{window}")
        if "std" in self.features_to_use:
            for window in range(1, 12):
                df[f"tn_std_{window}"] = df.groupby("product_id")["tn"].transform(lambda x: x.rolling(window=window, min_periods=1).std())
                self.features.append(f"tn_std_{window}")
    
        
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):
        from sklearn.linear_model import LinearRegression
        # si quiero  201912, entreno con 201812 (por estacionalidad)
        # lo busco dinamicamente con pred_df
        date_id_pred = pred_df["date_id"].unique()[0]
        train_df = train_df[train_df["date_id"] == (date_id_pred-12)]
        product_ids_magicos = self._get_product_ids(train_df)

        train_df = train_df[train_df["product_id"].isin(product_ids_magicos)]
        if self.especialidad != "all":
            train_df = train_df[train_df["cat1"] == self.especialidad]

        # elimino registros incompletos
        target = "target"
        train_df = train_df.dropna(subset=self.features + [target])
        print(f"Registros de entrenamiento: {len(train_df)}")
        X = train_df[self.features]
        y = train_df[target]
        model = LinearRegression()
        model.fit(X, y)
        self.model = model

        # hago la prediccion
        pred_df = pred_df.copy()
        
        # solo hace la prediccion para los productos que tienen todas las features
        pred_df = pred_df.dropna(subset=self.features)
        if self.especialidad != "all":
            pred_df = pred_df[pred_df["cat1"] == self.especialidad]
        X_pred = pred_df[self.features]
        pred_df["prediction"] = model.predict(X_pred).clip(min=0)  # Aseguro que la prediccion no sea negativa
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })
    

In [8]:
class LinearRegressionByProductModel:
    def __init__(self,):
        self.model = None
        self.features = []
    
    @property
    def name(self):
        return "LinearRegressionByProduct"

    
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum", "cat1": "first"}).sort_values(["product_id", "date_id"])
        self.features.append("tn")
        for lag in range(1, 24):
            df[f"tn_{lag}"] = df.groupby("product_id")["tn"].shift(lag)
            self.features.append(f"tn_{lag}") 
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):
        from sklearn.linear_model import LinearRegression
        # si quiero  201912, entreno con 201812 (por estacionalidad)
        # lo busco dinamicamente con pred_df
        date_id_pred = pred_df["date_id"].unique()[0]
        train_df = train_df[train_df["date_id"] == (date_id_pred-12)]

        models = {}
        for product_id in product_ids:
            product_id_train_df = train_df[train_df["product_id"] == product_id]
            target = "target"
            product_id_train_df = product_id_train_df.dropna(subset=[target])
            # hago fillna de features con el promedio del resto de las features en esa row en particular NO CON LA MEDIA DE TODO EL DATASET
            product_id_train_df[self.features] = product_id_train_df[self.features].fillna(
                product_id_train_df[self.features].mean(axis=0)
            ).fillna(0)
            if len(product_id_train_df) == 0:
                continue
            X = train_df[self.features]
            y = train_df[target]
            model = LinearRegression()
            model.fit(X, y)
            self.model = model
            models[product_id] = model        # hago la prediccion
        pred_df = pred_df.copy()
        
        pred_df["prediction"] = np.nan  # Inicializo la columna de predicciones
        for product_id in product_ids:
            model = models.get(product_id)
            if model is None:
                continue
            
            product_id_pred_df = pred_df[pred_df["product_id"] == product_id]
            
            # solo hace la prediccion para los productos que tienen todas las features
            #hago el fillna de features con el promedio del resto de las features en esa row en particular NO CON LA MEDIA DE TODO EL DATASET
            product_id_pred_df[self.features] = product_id_pred_df[self.features].fillna(
                product_id_pred_df[self.features].mean(axis=0)
            )
            X_pred = product_id_pred_df[self.features]
            y_pred = model.predict(X_pred).clip(min=0)
            pred_df.loc[pred_df["product_id"] == product_id, "prediction"] = y_pred
        
        # drop nan predictions
        pred_df = pred_df.dropna(subset=["prediction"])
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })
    

In [9]:
class SimpleMovingAveragePredictor:
    ''' Usa una media movil simple para predecir tn'''
    def __init__(self, window_size=12):
        self.model = None
        self.window_size = window_size

    @property
    def name(self):
        return f"SMA-{self.window_size}"
    
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum"}).sort_values(["product_id", "date_id"])
        # hago una columna que es la media movil simple agrupada por producto
        df["tn_sma"] = df.groupby("product_id")["tn"].transform(
            lambda x: x.rolling(window=self.window_size, min_periods=1).mean()
        )
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):

        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["tn_sma"],
        })

In [10]:
class ExponentialMovingAveragePredictor:
    ''' Usa una media movil exponencial para predecir tn'''
    def __init__(self, window_size=12):
        self.model = None
        self.window_size = window_size

    @property
    def name(self):
        return f"EMA-{self.window_size}"
    def prepare_dataset(self, df):
        df = df.copy()
        df = df.groupby(["product_id", "date_id"], as_index=False).agg({"tn": "sum", "target": "sum"}).sort_values(["product_id", "date_id"])
        # hago una columna que es la media movil exponencial agrupada por producto
        df["tn_ema"] = df.groupby("product_id")["tn"].transform(
            lambda x: x.ewm(span=self.window_size, adjust=False).mean()
        )
        return df
    
    def fit_and_predict(self, train_df, pred_df, *args, **kwargs):
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["tn_ema"],
        })

In [11]:
class AutoGluonPredictor:
    def __init__(self, presets="best_quality", estimator=None):
        self.model = None
        self.presets = presets
        self.estimator = estimator

    @property
    def name(self):
        if self.estimator:
            return f"AutoGluon-{self.estimator}"
        return f"AutoGluon-{self.presets}"
    
    def prepare_dataset(self, df):

        df = df.copy()
        df["fecha"] = df["fecha"].apply(lambda x: x.to_timestamp("M"))
        df = df.rename(columns={"fecha": "timestamp"})
        df["product_id"] = df["product_id"].astype(int)
        df["serie_id"] = df["product_id"].astype(str) + "-" + df["customer_id"].astype(str)
        df["cat1"] = df["cat1"].astype("category")
        df["cat2"] = df["cat2"].astype("category")
        df["cat3"] = df["cat3"].astype("category")
        df["brand"] = df["brand"].astype("category")
        df["sku_size"] = df["sku_size"].astype("category")

        self.static_features_df = pd.DataFrame({
            "cat1": df.groupby("serie_id")["cat1"].first(),
            "cat2": df.groupby("serie_id")["cat2"].first(),
            "cat3": df.groupby("serie_id")["cat3"].first(),
            "brand": df.groupby("serie_id")["brand"].first(),
            "sku_size": df.groupby("serie_id")["sku_size"].first(),
            "customer_id": df.groupby("serie_id")["customer_id"].first(),
            "product_id": df.groupby("serie_id")["product_id"].first(),
        }).reset_index()
        

        min_periods = 12  # Mínimo 6 meses de datos
        product_counts = df.groupby(["product_id"]).size()
        valid_products = product_counts[product_counts >= min_periods].index
        df = df[df["product_id"].isin(valid_products)]        
        df = df.dropna(subset=["tn"])
        return df
    
    def fit_and_predict(self, train_df, pred_df, df_model):
        from autogluon.timeseries import TimeSeriesPredictor, TimeSeriesDataFrame
        # el autogluon lo entreno con todas las fechas hasta pred_df
        train_df = df_model[df_model["date_id"] <= pred_df["date_id"].unique()[0]]
        train_df = train_df[train_df["product_id"].isin(product_ids)]
        ts_data = TimeSeriesDataFrame.from_data_frame(
            train_df.drop(columns=["target"]), 
            id_column="serie_id", 
            timestamp_column="timestamp", 
            static_features_df=self.static_features_df
        )
        ts_data = ts_data.sort_index()
        ts_data = ts_data.fill_missing_values()

        predictor = TimeSeriesPredictor(
            prediction_length=2,
            target="tn",
            freq="MS",
        )
        if self.estimator:
            predictor.fit(ts_data, hyperparameters={self.estimator: {}})
        else:
            predictor.fit(ts_data, presets=self.presets)
        forecast = predictor.predict(ts_data)
        forecast_mean = forecast["mean"].reset_index()
        forecast_mean = forecast_mean[forecast_mean["timestamp"] == forecast_mean["timestamp"].max()]
        forecast_mean[["product_id", "customer_id"]] = forecast_mean["item_id"].str.split("-", expand=True)
        forecast_mean["product_id"] = forecast_mean["product_id"].astype(int)
        forecast_mean = forecast_mean.groupby("product_id").agg({
            "mean": "sum",
        }).reset_index()

        # rename item_id to product_id
        pred_df = pred_df.copy()
        pred_df["product_id"] = pred_df["product_id"].astype(int)
        # separo serie_id en product_id y customer_id
        pred_df = pred_df.groupby(["product_id"]).agg({
            "target": "sum",
            "date_id": "first"
        }).reset_index()
        pred_df = pred_df.merge(forecast_mean, on=["product_id"], how="left")
        pred_df = pred_df.rename(columns={"mean": "prediction"})
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })



In [12]:


class BaseTabularPredictor:
    
    def _scaling_df(self, df, train=True):
        df = df.copy()
        import re
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        transformations = {
            "tn": [
                r"tn$",
                r"cust_request_qty_per_tn$",
                r"tn_lag_*",
                r"tn_rolling_mean_*",
                r"tn_rolling_max_*",
                r"tn_rolling_min_*",
                r"tn_.*_vendidas$",
                r"tn_agg*",
                r"tn_wavelet_*",
            ]
            + [r"stock_final$"]
            + [r"cust_request_tn_minus_tn$"]
            + [r"tn_diff_*"],
            "cust_request_qty": [
                r"cust_request_qty$",
                r"cust_request_qty_lag_*",
                r"cust_request_qty_rolling_mean_*",
                r"cust_request_qty_rolling_max_*",
                r"cust_request_qty_rolling_min_*",
                r"cust_request_qty_.*_vendidas$",
                r"cust_request_qty_agg*",
                r"cust_request_qty_wavelet_*",
            ]
            + [r"cust_request_qty_diff_*"],
        }

        # busco todas las columnas que empiezan con prod_ y agrego key y valor en transformation
        for col in numeric_cols:
            if col.startswith("prod_"):
                transformations[col] = [r"{}$".format(col)]

        from pandas.errors import PerformanceWarning
        import warnings
        warnings.simplefilter(action="ignore", category=PerformanceWarning)
        df = df.set_index(['serie_id', "date_id"])
        if train:
            prod_stats = df.groupby(["serie_id"])[
                list(transformations.keys())
            ].agg(["std"])
            prod_stats.columns = [
                f"{col[0]}_{col[1]}" for col in prod_stats.columns
            ]  # renombro las columnas para que no tengan tupla

            prod_stats = prod_stats.reset_index()
            self.prod_stats = prod_stats
            prod_stats = prod_stats.set_index(['serie_id'])
            # supress performance warnings
            self.prod_stats = prod_stats

            print("Scaling")
        else:
            if self.prod_stats is None:
                raise ValueError("prod_stats is not set. Call prepare_dataset first.")
            prod_stats = self.prod_stats
        for trainer, regex_cols in transformations.items():
            for col in regex_cols:
                matching_cols = [c for c in numeric_cols if re.match(col, c)]
                if not matching_cols:
                    continue
                for col in matching_cols:
                    std_col = prod_stats[trainer + "_std"]
                    df[f"{col}_scaled"] = (df[col] / std_col).replace([np.inf, -np.inf], np.nan)

        # scalo el target con tn_std
        df["target_scaled"] = df["target"] / prod_stats["tn_std"]
        df["target_scaled"] = df["target_scaled"].replace([np.inf, -np.inf], np.nan).fillna(0)

        df = df.reset_index()
        return df

    def prepare_dataset(self, df):
        df = df.copy()
        df["serie_id"] = df["product_id"].astype(str) + "-" + df["customer_id"].astype(str)
        return df


class AutoMLPredictor(BaseTabularPredictor):
    
    def __init__(self, estimator="lgbm", time_budget=60):
        self.model = None
        self.prod_stats = None
        self.estimator = estimator
        self.time_budget = time_budget

    @property
    def name(self):
        return f"AutoML-{self.estimator}-{self.time_budget}s"
    
    def custom_metric(self, X_val, y_val, estimator, labels, X_train, y_train, *args, **kwargs):
        y_pred = estimator.predict(X_val)
    
        temp_df = pd.DataFrame({
            "product_id": X_val["product_id"].values,
            "customer_id": X_val["customer_id"].values,
            "y_true": y_val,
            "y_pred": y_pred
        })
        temp_df["product_id"] = temp_df["product_id"].astype(int)
        temp_df["customer_id"] = temp_df["customer_id"].astype(int)
        prod_stats = self.prod_stats.copy().reset_index()
        prod_stats[["product_id", "customer_id"]] = prod_stats["serie_id"].str.split("-", expand=True)
        prod_stats["product_id"] = prod_stats["product_id"].astype(int)
        prod_stats["customer_id"] = prod_stats["customer_id"].astype(int)
        temp_df = temp_df.merge(prod_stats[["product_id", "customer_id", "tn_std"]], on=["product_id", "customer_id"], how="left")
        # desescale the predictions
        temp_df["y_pred"] = temp_df["y_pred"] * temp_df["tn_std"]
        temp_df["y_true"] = temp_df["y_true"] * temp_df["tn_std"]
    
        grouped = temp_df.groupby("product_id")[["y_true", "y_pred"]].sum()
        total_true = grouped["y_true"].sum()
    
        if total_true == 0:
            return 0.0, {"total_error": 0.0}
    
        total_error = np.abs(grouped["y_pred"] - grouped["y_true"]).sum() / total_true
        return total_error, {"total_error": total_error}

    def fit_and_predict(self, train_df, pred_df, df_model):
        from flaml import AutoML
        train_df = self._scaling_df(train_df, train=True)
        pred_df = self._scaling_df(pred_df, train=False)
        tscv = CustomTimeSeriesSplit(2, gap=1)
        automl_settings = {
            "time_budget": self.time_budget,
            "task": "regression",
            "metric": self.custom_metric,
            "estimator_list": [self.estimator],
            "n_jobs": -1,
            "eval_method": "cv",
            "split_type": tscv,
            "verbose": 3,
            "retrain_full": True
        }
        X_train = train_df.drop(columns=["target", "target_scaled", "fecha"])
        y_train = train_df["target_scaled"]
        automl = AutoML()
        automl.fit(X_train, y_train, **automl_settings)
        # hago la prediccion
        y_pred = automl.predict(pred_df.drop(columns=["target", "target_scaled", "fecha"]))
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()
 
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })
        


In [13]:
class AutoGluonTabularPredictor(BaseTabularPredictor):
    
    def __init__(self, presets="medium_quality", exclude_model_types=None, time_budget=None, subsample=1):
        self.model = None
        self.prod_stats = None
        self.presets = presets
        self.exclude_model_types = exclude_model_types or []
        self.time_budget = time_budget
        self.subsample = subsample

    @property
    def name(self):
        return f"AutoGluonTabular-{self.presets}-budget-{self.time_budget}s-subsample-{self.subsample}"
    
    def fit_and_predict(self, train_df, pred_df, df_model):
        from autogluon.tabular import TabularPredictor, TabularDataset
        train_df = self._scaling_df(train_df, train=True)
        pred_df = self._scaling_df(pred_df, train=False)

        # uso el date_id mas alto de train_df como tunning_data
        train_df = train_df.sample(frac=self.subsample, random_state=42)
        train = TabularDataset(train_df.drop(columns=["fecha", "serie_id"], errors='ignore'))
        pred = TabularDataset(pred_df.drop(columns=["fecha", "serie_id"], errors='ignore'))

        predictor = TabularPredictor(
            label="target_scaled",
            eval_metric="mean_absolute_error",
        )
        predictor.fit(
            train.drop(columns=["target"]),
            presets=self.presets,
            excluded_model_types=["RF", "XT"] + self.exclude_model_types,
            time_limit=self.time_budget,
        )

        y_pred = predictor.predict(pred.drop(columns=["target", "target_scaled"]))
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()
 
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })

In [14]:
class BasicXGBoostPredictor(BaseTabularPredictor):
    
    def __init__(self):
        self.model = None
        self.prod_stats = None

    @property
    def name(self):
        return f"BasicXGBoost"
    
    def fit_and_predict(self, train_df, pred_df, df_model):
        import xgboost as xgb
        train_df = self._scaling_df(train_df, train=True)
        pred_df = self._scaling_df(pred_df, train=False)

        X_train = train_df.drop(columns=["target", "target_scaled", "fecha", "serie_id"])
        y_train = train_df["target_scaled"]

        X_test = pred_df.drop(columns=["target", "target_scaled", "fecha", "serie_id"])
        y_test = pred_df["target_scaled"]
        
        # I use the tn_std as weight
        train_df = train_df.merge(self.prod_stats.reset_index()[["serie_id", "tn_std"]], on="serie_id", how="left")
        w_train = train_df["tn_std"].fillna(0)
        dtrain = xgb.DMatrix(X_train, label=y_train, weight=w_train, enable_categorical=True)
        dtest = xgb.DMatrix(X_test, label=y_test, enable_categorical=True)
        
        model = xgb.train(
            params={
                "objective": "reg:tweedie",
                "device": "cuda",
                "tree_method": "hist",
                "sampling_method": "uniform",
                "max_depth": 0,
                "learning_rate": 0.03,
                "num_leaves": 31,
                "subsample": 0.8,
                "colsample_bytree": 0.6,
            },
            dtrain=dtrain,
            evals=[(dtest, "test")],
            num_boost_round=1000,
            #num_boost_round=20
        )

        y_pred = model.predict(dtest)
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()
 
        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })

In [15]:
class BasicLGBMPredictor(BaseTabularPredictor):
    def __init__(self, with_scaling=True, extra_trees=False, n_trials=0, boosting_type="gbdt", use_weight=True, target="t+2"):
        self.model = None
        self.prod_stats = None
        self.with_scaling = with_scaling
        self.extra_trees = extra_trees
        self.n_trials = n_trials
        self.boosting_type = boosting_type
        self.base_params = {
            "objective": "tweedie",
            "device": "cpu",
            "max_bin": 512,
            "extra_trees": self.extra_trees,
            "boosting_type": self.boosting_type,
            "metric": "None",
        }
        self.use_weight = use_weight
        self.target = target # opciones: t+2, delta
        if self.target == "delta":
            self.base_params["objective"] = "regression"


    @property
    def name(self):
        return f"LGBM-extra_trees-{self.extra_trees}-trials-{self.n_trials}-scaling-{self.with_scaling}-boosting-{self.boosting_type}-weight-{self.use_weight}-target-{self.target}"


    def optimize_params(self,train_df):
        class LGBCustomMetric:
            def __init__(self, eval_df, prod_stats):
                self.eval_df = eval_df.copy()
                self.eval_df["serie_id"] = self.eval_df["product_id"].astype(str) + "-" + self.eval_df["customer_id"].astype(str)
                self.prod_stats = prod_stats

            def __call__(self, preds, train_data):
                eval_df = self.eval_df.copy()
                eval_df["prediction"] = preds

                eval_df = eval_df[eval_df["product_id"].isin(product_ids)]
                eval_df.set_index("serie_id", inplace=True)
                eval_df["prediction"] = eval_df["prediction"] * self.prod_stats["tn_std"]
                eval_df = eval_df.groupby("product_id").agg({
                    "target": "sum",
                    "prediction": "sum",
                }).reset_index()
                total_error = np.sum(np.abs(eval_df["prediction"] - eval_df["target"])) / np.sum(eval_df["target"])
                return "total_error", total_error, False  # False indicates that lower is better

        #uso la maxima fecha como eval_df
        eval_df = train_df[train_df["date_id"] == train_df["date_id"].max()].copy()
        train_df = train_df[train_df["date_id"] < eval_df["date_id"].max()].copy()
        from optuna import create_study
        from optuna.samplers import TPESampler
        import lightgbm as lgb
        def objective(trial):
            nonlocal train_df
            nonlocal eval_df
            train_df_trial = train_df.copy()
            eval_df_trial = eval_df.copy()
            if self.n_trials <= 1:
                print("Skipping hyperparameter optimization, using default parameters.")
                params = {
                    **self.base_params,
                    "num_leaves": 31,
                    "learning_rate": 0.03,
                    "feature_fraction": 0.8,
                    "bagging_fraction": 0.8,
                    "bagging_freq": 5,
                    "min_data_in_leaf": 30,
                }
            else:
                print("Optimizing hyperparameters with Optuna.")        
                params = {
                    **self.base_params,
                    "tweedie_variance_power": trial.suggest_float("tweedie_variance_power", 1.1, 1.9),
                    "num_leaves": trial.suggest_int("num_leaves", 16, 512),
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.075),
                    "feature_fraction": trial.suggest_float("feature_fraction", 0.3, 1.0),
                    "bagging_fraction": trial.suggest_float("bagging_fraction", 0.3, 1.0),
                    "bagging_freq": trial.suggest_int("bagging_freq", 1, 20),
                    "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 40),
                }
                if self.target == "delta":
                    # en este caso saco tweedie_variance_power porque no es necesario
                    params.pop("tweedie_variance_power", None)
            # TODO: optimizar custom metric porque relentiza mucho, mientras uso mae
            params["metric"] = "mae"
            # TODO: en la optimizacion NO Uso max bin por performance, pero en el entrenamiento si
            params["max_bin"] = 255
            # agrego los weights a train_df (y luego hago drop de la columna)
            train_df_trial = train_df_trial.merge(self.prod_stats.reset_index()[["serie_id", "tn_std"]], on="serie_id", how="left")
            w_train = train_df_trial["tn_std"].fillna(0)
            train_df_trial.drop(columns=["tn_std"], inplace=True)
            # si use_weight es False pongo w_train a 1.0
            if not self.use_weight:
                w_train = np.ones_like(w_train)

            dtrain = lgb.Dataset(train_df_trial.drop(columns=["target", "target_scaled", "fecha", "serie_id"]), label=train_df_trial["target_scaled"], weight=w_train)
            dval = lgb.Dataset(eval_df_trial.drop(columns=["target", "target_scaled", "fecha", "serie_id"]), label=eval_df_trial["target_scaled"])
            eval_result = {}
            if self.boosting_type == "gbdt":
                callbacks = [
                    lgb.log_evaluation(500),
                    lgb.early_stopping(int(400 + 4 / params["learning_rate"]), first_metric_only=True),
                    lgb.record_evaluation(eval_result),
                ]
            else:
                # dart no tiene early stopping
                callbacks = [
                    lgb.log_evaluation(500),
                    lgb.record_evaluation(eval_result),
                ]
            model = lgb.train(
                params, 
                dtrain, 
                num_boost_round=9999, 
                valid_sets=[dval], 
                valid_names=["eval"],
                #feval=LGBCustomMetric(eval_df, self.prod_stats),
                callbacks=callbacks
            )
            scores = eval_result["eval"]["l1"]
            best_iteration = min(enumerate(scores, 1), key=lambda x: x[1])[0]  # Encuentra la mejor iteración
            print(f"Best iteration: {best_iteration}, Score: {scores[best_iteration-1]}")
            # me guardo la mejor iteracion como attr del trial
            trial.set_user_attr("best_iteration", best_iteration)
            y_pred = model.predict(eval_df_trial.drop(columns=["target","target_scaled", "fecha", "serie_id"]), num_iteration=best_iteration)
            eval_df_trial["prediction"] = y_pred
            eval_df_trial.set_index("serie_id", inplace=True)
            eval_df_trial["prediction"] = eval_df_trial["prediction"] * self.prod_stats["tn_std"]
            if self.target == "delta":
                eval_df_trial["prediction"] = eval_df_trial["prediction"] + eval_df_trial["tn"]
                eval_df_trial["target"] = eval_df_trial["target"] + eval_df_trial["tn"]
            eval_df_trial = eval_df_trial.reset_index().groupby("product_id").agg({
                "target": "sum",
                "prediction": "sum",
            }).reset_index()
            total_error = np.sum(np.abs(eval_df_trial["prediction"] - eval_df_trial["target"])) / np.sum(eval_df_trial["target"])
            return total_error

        study = create_study(direction="minimize", sampler=TPESampler(seed=42))
        if self.n_trials == 0:
            self.n_trials = 1 # hago uno para optimizar al menos el n_estimator
        study.optimize(objective, n_trials=self.n_trials, show_progress_bar=True)
        best_trial = study.best_trial
        best_parameters = best_trial.params
        num_iteration = best_trial.user_attrs.get("best_iteration")
        if self.n_trials <= 1:
            return {
                "num_leaves": 31,
                "learning_rate": 0.03,
                "feature_fraction": 0.8,
                "bagging_fraction": 0.8,
                "bagging_freq": 5,
                "min_data_in_leaf": 30,
            }, num_iteration
        return best_parameters, num_iteration

    def fit_and_predict(self, train_df, pred_df, df_model):
        import lightgbm as lgb
        # si target es delta hago la diff entre target y tn
        if self.target == "delta":
            train_df["target"] = train_df["target"] - train_df["tn"]
            pred_df["target"] = pred_df["target"] - pred_df["tn"]
        if self.with_scaling:
            # Nota: puedo escalar con los datos de pred porque en el instante T tengo esos datos, lo que no tengo es el target
            train_scaling_df = df_model[df_model["date_id"] <= pred_df["date_id"].max()]
            train_scaling_df = self._scaling_df(train_scaling_df, train=True)
            train_df = self._scaling_df(train_df, train=False)
            pred_df = self._scaling_df(pred_df, train=False)
        else:
            # ignore SettingWithCopyWarning
            import warnings
            warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)
            self.prod_stats = pd.DataFrame({
                "serie_id": train_df["serie_id"].unique(),
                "tn_std": 1.0
            }).set_index("serie_id")
            train_df["target_scaled"] = train_df["target"]
            pred_df["target_scaled"] = pred_df["target"]

        best_params, num_iteration = self.optimize_params(train_df)
        X_train = train_df.drop(columns=["target", "target_scaled", "fecha", "serie_id"])
        y_train = train_df["target_scaled"]

        X_test = pred_df.drop(columns=["target", "target_scaled", "fecha", "serie_id"])
        y_test = pred_df["target_scaled"]
        
        # I use the tn_std as weight
        train_df = train_df.merge(self.prod_stats.reset_index()[["serie_id", "tn_std"]], on="serie_id", how="left")
        w_train = train_df["tn_std"].fillna(0)
        # si use_weight es False pongo w_train a 1.0
        if not self.use_weight:
            w_train = np.ones_like(w_train)
        
        dtrain = lgb.Dataset(X_train, label=y_train, weight=w_train)
        dtest = lgb.Dataset(X_test, label=y_test)

        params = {
            **self.base_params,
            **best_params,
            "verbose": 0,
        }
        print(f"Training LGBM with parameters: {params}, and {num_iteration} iterations")
        model = lgb.train(
            params,
            dtrain,
            #num_boost_round=1000,
            num_boost_round=num_iteration,
            valid_sets=[dtest],
            callbacks=[lgb.log_evaluation(1000)]
        )

        y_pred = model.predict(X_test)
        pred_df = pred_df.copy()
        pred_df["prediction"] = y_pred
        pred_df["serie_id"] = pred_df["product_id"].astype(str) + "-" + pred_df["customer_id"].astype(str)
        pred_df.set_index("serie_id", inplace=True)
        pred_df["prediction"] = pred_df["prediction"] * self.prod_stats["tn_std"]
        if self.target == "delta":
            # si target es delta, deshago la diff
            pred_df["prediction"] = pred_df["prediction"] + pred_df["tn"]
            pred_df["target"] = pred_df["target"] + pred_df["tn"]
        pred_df = pred_df.reset_index().groupby("product_id").agg({
            "target": "sum",
            "date_id": "first",
            "prediction": "sum"
        }).reset_index()

        return pd.DataFrame({
            "product_id": pred_df["product_id"],
            "date_id": pred_df["date_id"],
            "target": pred_df["target"],
            "prediction": pred_df["prediction"]
        })

In [16]:
#autogluon_tabular_predictor = AutoGluonTabularPredictor(presets="medium")
#df_autogluon_tabular = autogluon_tabular_predictor.prepare_dataset(df)
#test_df = df_autogluon_tabular[df_autogluon_tabular["date_id"] == 33]
#train_df = df_autogluon_tabular[df_autogluon_tabular["date_id"] < 32]
#results = autogluon_tabular_predictor.fit_and_predict(train_df, test_df, df_autogluon_tabular)

In [17]:

# import deep copy
from copy import deepcopy
class EnsambleTrainer:
    def __init__(self, models):
        self.models = models
        self.model_weights = None
        self.train_results = None

    def _combina_results(self, results):
        from collections import defaultdict
        # Diccionario para almacenar resultados intermedios
        combined_results = defaultdict(dict)

        for split, models in results.items():
            for model_info in models:
                model_name = model_info["model"].name
                pred_df = model_info["pred_df"]

                for _, row in pred_df.iterrows():
                    key = (row["product_id"], row["date_id"])
                    combined_results[key]["product_id"] = row["product_id"]
                    combined_results[key]["date_id"] = row["date_id"]
                    combined_results[key]["target"] = row["target"]
                    combined_results[key][f"prediction_{model_name}"] = row["prediction"]

        # Convertir a DataFrame
        results_df = pd.DataFrame(combined_results.values())

        # Opcional: ordenar columnas
        cols = ["product_id", "date_id", "target"] + sorted([col for col in results_df.columns if col not in {"product_id", "date_id", "target"}])
        results_df = results_df[cols]

        def fill_row_na_with_row_mean(row, prediction_cols):
            preds = row[prediction_cols]
            row[prediction_cols] = preds.fillna(preds.mean(skipna=True))
            return row

        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]

        results_df = results_df.apply(fill_row_na_with_row_mean, axis=1, prediction_cols=prediction_cols)   
        self.train_results = results_df
        return results_df
        
    def _optimize_weights(self, results_df, max_models=1):
        import numpy as np
        import pandas as pd
    
        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]
    
        model_weights_by_max = {}
    
        product_weights = {}
        predictions_list = []
    
        for _, row in results_df.iterrows():
            product_id = row["product_id"]
            target = row["target"]
    
            preds = np.array([row[col] for col in prediction_cols])
            diffs = preds - target
    
            if max_models == 1:
                # comportamiento original: one-hot del modelo más cercano
                errors = diffs ** 2
                best_idx = np.argmin(errors)
                weights = np.zeros(len(prediction_cols))
                weights[best_idx] = 1
            else:
                above = np.where(diffs >= 0)[0]
                below = np.where(diffs < 0)[0]
    
                if len(above) > 0 and len(below) > 0:
                    i_above = above[np.argmin(diffs[above])]
                    i_below = below[np.argmax(diffs[below])]
                    p1, p2 = preds[i_below], preds[i_above]
    
                    # resolver w en target = w*p1 + (1-w)*p2 => w = (target - p2)/(p1 - p2)
                    denom = p1 - p2
                    if denom != 0:
                        w = (target - p2) / denom
                        w = np.clip(w, 0, 1)
                    else:
                        w = 0.5  # predicciones iguales => promedio
    
                    weights = np.zeros(len(prediction_cols))
                    weights[i_below] = w
                    weights[i_above] = 1 - w
                else:
                    # todos arriba o todos abajo: fallback a comportamiento original
                    errors = diffs ** 2
                    best_idx = np.argmin(errors)
                    weights = np.zeros(len(prediction_cols))
                    weights[best_idx] = 1
    
            if product_id not in product_weights:
                product_weights[product_id] = []
            product_weights[product_id].append(weights)
    
        weights_list = []
        predictions_list = []
        product_ids = list(product_weights.keys())
    
        for product_id in product_ids:
            weight_arr = np.mean(product_weights[product_id], axis=0)
            weights_dict = dict(zip(prediction_cols, weight_arr))
            weights_list.append(weights_dict)
    
            product_rows = results_df[results_df["product_id"] == product_id]
            preds_matrix = product_rows[prediction_cols].values
            weighted_preds = preds_matrix @ weight_arr
            mean_prediction = np.mean(weighted_preds)
            predictions_list.append(mean_prediction)
    
        agg_df = pd.DataFrame({
            "product_id": product_ids,
            "weights": weights_list,
            "predictions": predictions_list
        })
    
        if not hasattr(self, "model_weights") or self.model_weights is None:
            self.model_weights = {}
    
        self.model_weights[max_models] = agg_df[["product_id", "weights"]].set_index("product_id")


    def _compute_metrics_simple(self, y_true, y_pred):
        """Calcula el error absoluto medio entre y_true e y_pred"""
        return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true) if np.sum(y_true) > 0 else 0

    def _compute_metrics(self, results_df, max_models=1):
        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]
        agg_df = results_df.groupby("product_id")[["target"] + prediction_cols].sum().reset_index()
        agg_df = agg_df.set_index("product_id")
        agg_df["weights"] = self.model_weights[max_models]["weights"]
        agg_df["prediction_ensamble"] = agg_df.apply(
            lambda row: sum(row[col] * row["weights"][col] for col in prediction_cols), 
            axis=1
        )
        self.agg_df = agg_df
        # calculo metricas
        metrics = {}
        def total_error(y_true, y_pred):
            return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

        prediction_cols = [col for col in agg_df.columns if col.startswith("prediction_")]
        for col in prediction_cols:
            metrics[col] = total_error(agg_df["target"], agg_df[col])
        return pd.DataFrame(metrics, index=[0]).T.rename(columns={0: "error"})
    
    def fit(self, df, splitter):
        """Entrena todos los modelos en cada split del splitter"""
        df = df.dropna(subset=["target"])
        number_of_splits = splitter.get_n_splits(df)
        results = {f"split_{i}": [] for i in range(number_of_splits)}
        for i, (train_idx, test_idx) in enumerate(splitter.split(df)):
            # Obtener las fechas de los splits originales
            train_dates = df.iloc[train_idx]["date_id"].unique()
            test_dates = df.iloc[test_idx]["date_id"].unique()
            
            for m in self.models:
                model = deepcopy(m)
                df_model = model.prepare_dataset(df)
                
                # Recalcular train/test usando las fechas, no los índices
                train_df = df_model[df_model["date_id"].isin(train_dates)]
                test_df = df_model[df_model["date_id"].isin(test_dates)]
                test_df = test_df[test_df["product_id"].isin(product_ids)]
                
                pred_df = model.fit_and_predict(train_df, test_df, df_model)
                results[f"split_{i}"].append({
                    "model": model,
                    "target": test_df[["target", "product_id", "date_id"]],
                    "pred_df": pred_df
                })
                print(f"Modelo {model.name} entrenado en split {i+1}/{number_of_splits}:")
                print(self._compute_metrics_simple(pred_df["target"], pred_df["prediction"]))
        
        results_df = self._combina_results(results)
        # VALIDACION
        biggest_date = results_df["date_id"].max()
        val_weights_df = results_df[results_df["date_id"] != biggest_date]
        test_df = results_df[results_df["date_id"] == biggest_date]
        self._optimize_weights(val_weights_df)
        print("VALIDACIÓN 1 MODEL MAX (primer fold con pesos del resto):")
        print(self._compute_metrics(test_df))

        self._optimize_weights(val_weights_df, max_models=2)
        print("VALIDACIÓN 2 MODEL MAX (primer fold con pesos del resto):")
        print(self._compute_metrics(test_df, max_models=2))

        
        self._optimize_weights(results_df)
        print(self._compute_metrics(results_df))
        self._optimize_weights(results_df, max_models=2)
        print(self._compute_metrics(results_df, max_models=2))

    def final_pred(self, df, kaggle_date_id):
        """Vuelve a entrenar todos los modelos con el dataset completo"""
        results = {"split_final": []}
        for m in self.models:
            model = deepcopy(m)
            df_model = model.prepare_dataset(df)
            # Recalcular train/test usando las fechas, no los índices
            train_df = df_model[df_model["date_id"] < kaggle_date_id]
            train_df = train_df.dropna(subset=["target"])
            pred_df = df_model[df_model["date_id"] == kaggle_date_id]
            pred_df = pred_df[pred_df["product_id"].isin(product_ids)]

            pred_df = model.fit_and_predict(train_df, pred_df, df_model)
            results["split_final"].append({
                "model": model,
                "target": pred_df[["target", "product_id", "date_id"]],
                "pred_df": pred_df
            })
        results_df = self._combina_results(results)  
        prediction_cols = [col for col in results_df.columns if col.startswith("prediction_")]
        agg_df = results_df.groupby("product_id")[["target"] + prediction_cols].sum().reset_index()
        agg_df = agg_df.set_index("product_id", drop=False)
        agg_df["weights"] = self.model_weights[1]["weights"]
        agg_df["prediction_ensamble"] = agg_df.apply(
            lambda row: sum(row[col] * row["weights"][col] for col in prediction_cols), 
            axis=1
        )
        agg_df["weights"] = self.model_weights[2]["weights"]
        agg_df["prediction_ensamble_2"] = agg_df.apply(
            lambda row: sum(row[col] * row["weights"][col] for col in prediction_cols), 
            axis=1
        )
        return agg_df.rename(columns={"prediction_ensamble": "tn", "prediction_ensamble_2": "tn_2"})


In [18]:
# TODO: entrenar el lightgbm con todos los product_ids? (la validacion solo con los 780)
import sys
from contextlib import redirect_stdout

#trainer = EnsambleTrainer([
#    #AutoGluonTabularPredictor(exclude_model_types=["KNN"]),
#    #BasicLGBMPredictor(n_trials=35, with_scaling=True, extra_trees=False, boosting_type="gbdt"),
#    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="gbdt"),
#    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="gbdt"),
#    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=True, boosting_type="gbdt"),
#    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="gbdt"),
#    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="dart"),
#    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="dart"),
#    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=True, boosting_type="dart"),
#    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="dart", use_weight=False),
#    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="gbdt", use_weight=False),
#    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="gbdt", use_weight=False),
#    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="dart", use_weight=False),
#    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="dart", use_weight=False),
#    #BasicXGBoostPredictor(),
#    LinearRegressionModel(),
#    #AutoGluonTabularPredictor(presets="best", time_budget=3600, exclude_model_types=["KNN"]),
#    AutoGluonPredictor(presets="best_quality"),
#    #AutoGluonPredictor(presets="fast_training"),
#    SimpleMovingAveragePredictor(window_size=12)
#])

# only linear regressions + moving average
trainer = EnsambleTrainer([
    LinearRegressionModel(),
    LinearRegressionModel(especialidad="HC"),
    LinearRegressionModel(especialidad="FOODS"),
    LinearRegressionModel(especialidad="PC"),
    LinearRegressionModel(product_ids="all"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="gbdt", use_weight=True, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=True, boosting_type="gbdt", use_weight=True, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="gbdt", use_weight=True, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="gbdt", use_weight=True, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="dart", use_weight=True, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=True, boosting_type="dart", use_weight=True, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="dart", use_weight=True, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="dart", use_weight=True, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="gbdt", use_weight=False, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=True, boosting_type="gbdt", use_weight=False, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="gbdt", use_weight=False, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="gbdt", use_weight=False, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="dart", use_weight=False, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=True, boosting_type="dart", use_weight=False, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="dart", use_weight=False, target="delta"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="dart", use_weight=False, target="delta"),


    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="gbdt", use_weight=True, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=True, boosting_type="gbdt", use_weight=True, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="gbdt", use_weight=True, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="gbdt", use_weight=True, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="dart", use_weight=True, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=True, boosting_type="dart", use_weight=True, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="dart", use_weight=True, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="dart", use_weight=True, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="gbdt", use_weight=False, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=True, boosting_type="gbdt", use_weight=False, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="gbdt", use_weight=False, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="gbdt", use_weight=False, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=True, boosting_type="dart", use_weight=False, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=True, boosting_type="dart", use_weight=False, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=True, extra_trees=False, boosting_type="dart", use_weight=False, target="t+2"),
    BasicLGBMPredictor(n_trials=0, with_scaling=False, extra_trees=False, boosting_type="dart", use_weight=False, target="t+2"),
    AutoGluonPredictor(presets="best_quality"),
    AutoGluonPredictor(presets="fast_training"), # only stadistical models
    ##LinearRegressionByProductModel(),
    SimpleMovingAveragePredictor(window_size=12),
])
splitter = CustomTimeSeriesSplit(n_splits=3, gap=1)
trainer.fit(df, splitter)


Registros de entrenamiento: 33
Modelo LinearRegression-magicos-['lags']-all entrenado en split 1/3:
0.33242905
Registros de entrenamiento: 20
Modelo LinearRegression-magicos-['lags']-HC entrenado en split 1/3:
0.4803329
Registros de entrenamiento: 5
Modelo LinearRegression-magicos-['lags']-FOODS entrenado en split 1/3:
0.3568
Registros de entrenamiento: 8
Modelo LinearRegression-magicos-['lags']-PC entrenado en split 1/3:
0.3492257
Registros de entrenamiento: 786
Modelo LinearRegression-all-['lags']-all entrenado en split 1/3:
0.27242267


/tmp/ipykernel_63886/1139988248.py:163: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["target"] = train_df["target"] - train_df["tn"]


Scaling


/home/fede/programacion/labo3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-07-19 14:46:30,900] A new study created in memory with name: no-name-3afe1fbe-c0bb-412e-9fa7-b7e2ac2cbea4
  0%|          | 0/1 [00:00<?, ?it/s]

Skipping hyperparameter optimization, using default parameters.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.050934 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 141489
[LightGBM] [Info] Number of data points in the train set: 26342, number of used features: 575
[LightGBM] [Info] Start training from score -0.047139
Training until validation scores don't improve for 533 rounds
[500]	eval's l1: 0.761126
[1000]	eval's l1: 0.749115
[1500]	eval's l1: 0.743917
[2000]	eval's l1: 0.736729
[2500]	eval's l1: 0.73078
[3000]	eval's l1: 0.727
[3500]	eval's l1: 0.722417
[4000]	eval's l1: 0.720989
[4500]	eval's l1: 0.719199


...

Modelo LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-t+2 entrenado en split 3/3:
0.4457316167653073


Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250720_005852'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jun 26 14:16:59 UTC 2
CPU Count:          16
GPU Count:          1
Memory Avail:       17.00 GB / 31.23 GB (54.4%)
Disk Space Avail:   725.41 GB / 914.78 GB (79.3%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'verbosity': 2}

train_data with frequency 'ME' has been resampled

Modelo AutoGluon-best_quality entrenado en split 3/3:
0.32437684220687396


Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250720_010834'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jun 26 14:16:59 UTC 2
CPU Count:          16
GPU Count:          1
Memory Avail:       15.56 GB / 31.23 GB (49.8%)
Disk Space Avail:   725.01 GB / 914.78 GB (79.3%)
Setting presets to: fast_training

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'MS',
 'hyperparameters': 'very_light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'verbosity': 2}

train_data with frequency 'ME' has been resam

Modelo AutoGluon-fast_training entrenado en split 3/3:
0.2807924484712074
Modelo SMA-12 entrenado en split 3/3:
0.251847995358391
VALIDACIÓN 1 MODEL MAX (primer fold con pesos del resto):
                                                       error
prediction_AutoGluon-best_quality                   0.285887
prediction_AutoGluon-fast_training                  0.260927
prediction_LGBM-extra_trees-False-trials-1-scal...  0.290678
prediction_LGBM-extra_trees-False-trials-1-scal...  0.253521
prediction_LGBM-extra_trees-False-trials-1-scal...  0.290678
prediction_LGBM-extra_trees-False-trials-1-scal...  0.255729
prediction_LGBM-extra_trees-False-trials-1-scal...  0.297885
prediction_LGBM-extra_trees-False-trials-1-scal...  0.253691
prediction_LGBM-extra_trees-False-trials-1-scal...  0.297885
prediction_LGBM-extra_trees-False-trials-1-scal...  0.253691
prediction_LGBM-extra_trees-False-trials-1-scal...  0.261849
prediction_LGBM-extra_trees-False-trials-1-scal...  0.243467
prediction_LGBM-ext

In [19]:
trainer.model_weights[1]

,weights
product_id,
20001.0,"{'prediction_AutoGluon-best_quality': 0.0, 'pr..."
20002.0,"{'prediction_AutoGluon-best_quality': 0.0, 'pr..."
20003.0,{'prediction_AutoGluon-best_quality': 0.333333...
20004.0,"{'prediction_AutoGluon-best_quality': 0.0, 'pr..."
20005.0,"{'prediction_AutoGluon-best_quality': 0.0, 'pr..."
...,...
21252.0,"{'prediction_AutoGluon-best_quality': 0.0, 'pr..."
21265.0,"{'prediction_AutoGluon-best_quality': 0.0, 'pr..."
21266.0,{'prediction_AutoGluon-best_quality': 0.333333...


In [20]:
trainer._compute_metrics(trainer.train_results)

,error
prediction_AutoGluon-best_quality,0.158927
prediction_AutoGluon-fast_training,0.147683
prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-delta,0.145159
prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-t+2,0.248488
prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-delta,0.144192
prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-t+2,0.242862
prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-delta,0.153596
prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2,0.203961
prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-delta,0.153596
prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-t+2,0.203961


In [21]:

models_used = [model.name for model in trainer.models]
models_used = " ".join(models_used)
# hago un hash en base de models_used para el nombre del archivo
import hashlib
hash_object = hashlib.md5(models_used.encode())
hash_hex = hash_object.hexdigest()

trainer.train_results.to_csv(f"train_results_{hash_hex}.csv", index=False)

In [22]:
trainer.agg_df

,target,prediction_AutoGluon-best_quality,prediction_AutoGluon-fast_training,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-delta,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-delta,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-delta,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-True-target-delta,...,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-True-target-delta,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2,prediction_LinearRegression-all-['lags']-all,prediction_LinearRegression-magicos-['lags']-FOODS,prediction_LinearRegression-magicos-['lags']-HC,prediction_LinearRegression-magicos-['lags']-PC,prediction_LinearRegression-magicos-['lags']-all,prediction_SMA-12,weights,prediction_ensamble
product_id,,,,,,,,,,,,,,,,,,,,,
20001.0,4463.566406,4192.395459,4372.735319,4422.990425,3378.876181,4396.318820,3405.254866,4323.707452,3221.895933,4323.707452,...,4171.962970,4103.809705,4702.397339,4043.273878,4340.302307,4043.273878,4933.541870,4567.446299,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",4284.419268
20002.0,4490.422363,3117.013740,3132.103425,3775.911540,3068.031680,3758.201114,3075.186824,3670.705772,3133.139903,3670.705772,...,3168.781488,3182.906228,4227.423950,3314.785060,4087.703308,3314.785060,4550.385742,3481.152552,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",4008.257663
20003.0,2922.161682,2693.429281,2570.598216,2212.192339,2043.189654,2239.980674,2016.316762,2096.693808,2130.119467,2096.693808,...,2375.259509,2316.656912,3170.063721,2680.570129,2366.516722,2366.516722,3459.910645,2422.960470,{'prediction_AutoGluon-best_quality': 0.333333...,2790.503471
20004.0,2426.538391,1933.518164,2141.002004,1758.915773,1470.870708,1780.698198,1480.297685,1721.726093,1587.317701,1721.726093,...,1833.624492,1766.360850,1775.895447,1797.609314,1762.877578,1762.877578,2037.894104,1859.216726,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1992.168474
20005.0,2196.938965,1812.241342,1992.391091,1874.755538,1276.263161,1870.320513,1280.920390,1727.971250,1369.488550,1727.971250,...,1790.897510,1602.736698,1653.824432,1472.110077,1654.632991,1654.632991,2059.518066,1888.259326,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1692.873893
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21263.0,0.060690,0.163030,0.058352,0.604628,0.058426,0.579401,0.053004,0.284404,0.754035,0.284404,...,0.061787,0.115338,0.150209,0.180201,0.180201,0.238204,0.150209,0.285505,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.285586
21265.0,0.225280,0.325172,0.325172,0.372250,0.176697,0.399770,0.156400,0.537846,0.846566,0.537846,...,0.194472,0.190116,0.325172,0.325172,0.325172,0.325172,0.325172,0.301994,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.218462
21266.0,0.236650,0.336069,0.336069,0.383420,0.173076,0.407427,0.155701,0.381304,0.843787,0.381304,...,0.223481,0.188721,0.336069,0.336069,0.336069,0.336069,0.336069,0.318535,{'prediction_AutoGluon-best_quality': 0.333333...,0.264897


In [23]:
final_df = trainer.final_pred(df, 35)

Registros de entrenamiento: 33
Registros de entrenamiento: 20
Registros de entrenamiento: 5
Registros de entrenamiento: 8
Registros de entrenamiento: 791
Scaling


[I 2025-07-19 22:09:10,228] A new study created in memory with name: no-name-bf9737ba-59b4-4459-8cee-765b4b956f62
  0%|          | 0/1 [00:00<?, ?it/s]

Skipping hyperparameter optimization, using default parameters.
Training until validation scores don't improve for 533 rounds
[500]	eval's l1: 0.631529
[1000]	eval's l1: 0.619581
[1500]	eval's l1: 0.615812
[2000]	eval's l1: 0.611223
[2500]	eval's l1: 0.611427
[3000]	eval's l1: 0.609331
[3500]	eval's l1: 0.608662
[4000]	eval's l1: 0.607108
[4500]	eval's l1: 0.607433


Best trial: 0. Best value: 0.273904: 100%|██████████| 1/1 [01:52<00:00, 112.34s/it]

Early stopping, best iteration is:
[4091]	eval's l1: 0.606633
Evaluated only: l1
Best iteration: 4091, Score: 0.6066331333290785
[I 2025-07-19 22:11:02,571] Trial 0 finished with value: 0.27390361560799503 and parameters: {}. Best is trial 0 with value: 0.27390361560799503.
Training LGBM with parameters: {'objective': 'regression', 'device': 'cpu', 'max_bin': 512, 'extra_trees': True, 'boosting_type': 'gbdt', 'metric': 'None', 'num_leaves': 31, 'learning_rate': 0.03, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'min_data_in_leaf': 30, 'verbose': 0}, and 4091 iterations


...

[I 2025-07-20 00:21:09,588] A new study created in memory with name: no-name-421159dd-1590-4dcd-8c96-7507935e2ba6
  0%|          | 0/1 [00:00<?, ?it/s]

Skipping hyperparameter optimization, using default parameters.
[500]	eval's l1: 9.06898
[1000]	eval's l1: 8.47198
[1500]	eval's l1: 8.23612
[2000]	eval's l1: 7.67264
[2500]	eval's l1: 7.54379
[3000]	eval's l1: 7.4882
[3500]	eval's l1: 7.43753
[4000]	eval's l1: 7.42397
[4500]	eval's l1: 7.45165
[5000]	eval's l1: 7.46178
[5500]	eval's l1: 7.47442
[6000]	eval's l1: 7.47142
[6500]	eval's l1: 7.4593
[7000]	eval's l1: 7.48594
[7500]	eval's l1: 7.49713
[8000]	eval's l1: 7.54331
[8500]	eval's l1: 7.52573
[9000]	eval's l1: 7.49906
[9500]	eval's l1: 7.47868


Best trial: 0. Best value: 0.931817: 100%|██████████| 1/1 [03:08<00:00, 188.81s/it]

Best iteration: 3629, Score: 7.356593310958047
[I 2025-07-20 00:24:18,402] Trial 0 finished with value: 0.9318166813807647 and parameters: {}. Best is trial 0 with value: 0.9318166813807647.
Training LGBM with parameters: {'objective': 'tweedie', 'device': 'cpu', 'max_bin': 512, 'extra_trees': False, 'boosting_type': 'dart', 'metric': 'None', 'num_leaves': 31, 'learning_rate': 0.03, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'min_data_in_leaf': 30, 'verbose': 0}, and 3629 iterations



Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250720_032605'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #29~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Thu Jun 26 14:16:59 UTC 2
CPU Count:          16
GPU Count:          1
Memory Avail:       16.80 GB / 31.23 GB (53.8%)
Disk Space Avail:   724.94 GB / 914.78 GB (79.2%)
Setting presets to: best_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'MS',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'verbosity': 2}

train_data with frequency 'ME' has been resample

In [28]:
final_df

,product_id,target,prediction_AutoGluon-best_quality,prediction_AutoGluon-fast_training,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-delta,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-False-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-delta,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-dart-weight-True-target-t+2,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-delta,prediction_LGBM-extra_trees-False-trials-1-scaling-False-boosting-gbdt-weight-False-target-t+2,...,prediction_LGBM-extra_trees-True-trials-1-scaling-True-boosting-gbdt-weight-True-target-t+2,prediction_LinearRegression-all-['lags']-all,prediction_LinearRegression-magicos-['lags']-FOODS,prediction_LinearRegression-magicos-['lags']-HC,prediction_LinearRegression-magicos-['lags']-PC,prediction_LinearRegression-magicos-['lags']-all,prediction_SMA-12,weights,tn,tn_2
product_id,,,,,,,,,,,,,,,,,,,,,
20001.0,20001.0,0.0,1315.988068,1414.626422,1625.705279,1181.747647,1625.705279,1198.199425,1590.005865,1367.074970,...,1271.386618,1277.502197,1364.332970,1199.431152,1364.332970,1162.707520,1454.732737,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1393.236927,1381.724773
20002.0,20002.0,0.0,1096.504174,1118.941426,1166.179259,1167.432187,1166.179259,1194.259037,1198.439143,1346.510526,...,1345.176336,1222.369629,1239.921424,1294.223633,1239.921424,1183.640625,1175.437134,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",1203.750208,1208.116727
20003.0,20003.0,0.0,693.700877,746.214679,610.529758,658.460728,610.529758,628.172522,621.712087,632.810920,...,768.385407,765.707947,643.368164,743.975282,743.975282,684.763855,784.976405,{'prediction_AutoGluon-best_quality': 0.199020...,755.089894,748.838745
20004.0,20004.0,0.0,521.957952,565.213435,629.887948,557.094131,629.887948,542.904335,548.114435,544.855566,...,594.886968,604.437927,501.384521,581.107301,581.107301,580.485046,627.215322,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",549.027668,556.595002
20005.0,20005.0,0.0,486.702169,555.912126,661.317679,547.798851,661.317679,562.972249,567.994537,548.287994,...,581.862732,568.349304,441.741516,576.480911,576.480911,563.560852,668.270111,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",612.534548,614.142580
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21263.0,21263.0,0.0,-0.000782,0.006956,1.229206,0.018691,1.229206,0.023327,1.404239,0.016714,...,0.015996,1.899233,0.331520,0.331520,0.619479,0.467764,0.029993,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.012265,0.013266
21265.0,21265.0,0.0,0.274838,0.274838,0.857413,0.043563,0.857413,0.041426,1.131591,0.044660,...,0.063152,0.274838,0.274838,0.274838,0.274838,0.274838,0.089541,"{'prediction_AutoGluon-best_quality': 0.0, 'pr...",0.058889,0.063042
21266.0,21266.0,0.0,0.285134,0.285134,0.944235,0.045977,0.944235,0.043808,1.132731,0.044603,...,0.061001,0.285134,0.285134,0.285134,0.285134,0.285134,0.094659,{'prediction_AutoGluon-best_quality': 0.192409...,0.138368,0.109723


In [29]:

models_used = [model.name for model in trainer.models]
models_used = " ".join(models_used)
# hago un hash en base de models_used para el nombre del archivo
import hashlib
hash_object = hashlib.md5(models_used.encode())
hash_hex = hash_object.hexdigest()
description = f"Ensamble de modelos: {models_used}"
# save txt with name hash_hex.txt and the description
with open(f"description_{hash_hex}.txt", "w") as f:
    f.write(description)
submission = final_df[["product_id", "tn"]].reset_index(drop=True)
submission.to_csv(f"submission_weighted_ensamble_{hash_hex}.csv", index=False)
submission

,product_id,tn
0,20001.0,1393.236927
1,20002.0,1203.750208
2,20003.0,755.089894
3,20004.0,549.027668
4,20005.0,612.534548
...,...,...
775,21263.0,0.012265
776,21265.0,0.058889
777,21266.0,0.138368
778,21267.0,0.044595


In [34]:
final_df.to_csv(f"final_df_{hash_hex}.csv", index=False)
hash_hex

'08f7f3fe2145910b0357d179691f8c60'

In [30]:
submission_2_models = final_df[["product_id", "tn_2"]].reset_index(drop=True)
submission_2_models.rename(columns={"tn_2": "tn"}, inplace=True)
submission_2_models.to_csv(f"submission_2_models_weighted_ensamble_{hash_hex}.csv", index=False)
submission_2_models

,product_id,tn
0,20001.0,1381.724773
1,20002.0,1208.116727
2,20003.0,748.838745
3,20004.0,556.595002
4,20005.0,614.142580
...,...,...
775,21263.0,0.013266
776,21265.0,0.063042
777,21266.0,0.109723
778,21267.0,0.177618


In [31]:
print(description)

Ensamble de modelos: LinearRegression-magicos-['lags']-all LinearRegression-magicos-['lags']-HC LinearRegression-magicos-['lags']-FOODS LinearRegression-magicos-['lags']-PC LinearRegression-all-['lags']-all LGBM-extra_trees-True-trials-0-scaling-True-boosting-gbdt-weight-True-target-delta LGBM-extra_trees-True-trials-0-scaling-False-boosting-gbdt-weight-True-target-delta LGBM-extra_trees-False-trials-0-scaling-True-boosting-gbdt-weight-True-target-delta LGBM-extra_trees-False-trials-0-scaling-False-boosting-gbdt-weight-True-target-delta LGBM-extra_trees-True-trials-0-scaling-True-boosting-dart-weight-True-target-delta LGBM-extra_trees-True-trials-0-scaling-False-boosting-dart-weight-True-target-delta LGBM-extra_trees-False-trials-0-scaling-True-boosting-dart-weight-True-target-delta LGBM-extra_trees-False-trials-0-scaling-False-boosting-dart-weight-True-target-delta LGBM-extra_trees-True-trials-0-scaling-True-boosting-gbdt-weight-False-target-delta LGBM-extra_trees-True-trials-0-scalin

In [32]:
submission_3 = final_df_3[["product_id", "pred_weights"]].rename(columns={"pred_weights": "tn"}).reset_index(drop=True)
submission_3.to_csv(f"submission_weights_ensamble_per_product_{hash_hex}.csv", index=False)
submission_3

NameError: name 'final_df_3' is not defined